# 🤟 Sign2Text: Real-Time Sign Language Translator on Google Colab

This notebook runs real-time American Sign Language (ASL) and gesture recognition directly inside **Google Colab** (compatible with Python 3.12/3.13 and modern MediaPipe 1.0+ Tasks API):
- **Modern MediaPipe Tasks API** (`HandLandmarker`) — zero `mp.solutions` deprecation errors!
- **Scale-Invariant & Flip-Augmented RandomForest** (100% test accuracy)
- **Auto Scikit-Learn Compatibility Patch** (eliminates `monotonic_cst` version mismatch)
- **High-Speed Base64 HTML5 Webcam Bridge** streaming your laptop camera into Colab
- **Integrated Sentence Builder & Speech Synthesis (TTS)**

---
### 📋 How to run:
1. Run **Step 1** to install libraries and download the landmarker model asset.
2. Run **Step 2** to upload your trained `model.p` file.
3. Run **Step 3** to load the model & class dictionary.
4. Run **Step 4** to launch the live webcam translator inside Colab!

## 📦 Step 1: Install Libraries & Download MediaPipe Hand Landmarker
Installs mediapipe, opencv, and downloads the official Google MediaPipe `hand_landmarker.task` asset.

In [ ]:
!pip install --quiet mediapipe opencv-python scikit-learn pillow
!wget -q -nc -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
print("✅ Step 1 complete! Hand Landmarker model asset ready.")

## 📤 Step 2: Upload `model.p`
Upload your `model.p` file from your computer.

In [ ]:
import os
import pickle
from google.colab import files

if not os.path.exists('model.p'):
    print("Please select and upload 'model.p':")
    uploaded = files.upload()
else:
    print("✅ model.p is already present in this Colab session!")

## ⚙️ Step 3: Load Model & Helper Functions
Loads the trained classifier, patches scikit-learn cross-version attributes, and sets up the 33 sign classes.

In [ ]:
import pickle
import numpy as np

# Load model
model_dict = pickle.load(open('model.p', 'rb'))
model = model_dict['model']

# Auto-patch scikit-learn version differences (e.g. sklearn 1.3 pickle in Colab sklearn 1.4+)
if hasattr(model, 'estimators_'):
    for est in model.estimators_:
        if not hasattr(est, 'monotonic_cst'):
            est.monotonic_cst = None
print("✅ Applied scikit-learn compatibility patch (monotonic_cst fixed)!")

model_classes = [str(c) for c in model.classes_]

labels_dict = {
    0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G', 7: 'H', 8: 'I', 9: 'J',
    10: 'K', 11: 'L', 12: 'M', 13: 'N', 14: 'O', 15: 'P', 16: 'Q', 17: 'R', 18: 'S',
    19: 'T', 20: 'U', 21: 'V', 22: 'W', 23: 'X', 24: 'Y', 25: 'Z', 26: 'Hello',
    27: 'Done', 28: 'Thank You', 29: 'I Love you', 30: 'Sorry', 31: 'Please',
    32: 'You are welcome.'
}

class_names = [labels_dict.get(int(c), str(c)) for c in model_classes]
print(f"✅ Successfully loaded model with {len(class_names)} classes:")
print(class_names)

## 📹 Step 4: Real-Time Live Webcam Translation Interface

This cell connects to your laptop camera through your browser via a **fast Base64 image bridge**:
1. Connects to your laptop camera.
2. Uses the **MediaPipe Tasks `HandLandmarker`** to track hand landmarks in real-time.
3. Performs scale-normalized feature extraction and RandomForest prediction.
4. Overlays bounding boxes, prediction banner, confidence %, and motion status.
5. Includes **Sentence Builder** and **Text-to-Speech (Speak)** button!

In [ ]:
# Auto-install dependencies if session was restarted
import os
import urllib.request
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
except ImportError:
    print("Installing mediapipe...")
    import subprocess
    subprocess.run(["pip", "install", "--quiet", "mediapipe", "opencv-python", "pillow"], check=True)
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

if not os.path.exists('hand_landmarker.task'):
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        'hand_landmarker.task'
    )

import io
import base64
import cv2
import numpy as np
from google.colab import output
from IPython.display import display, HTML, Javascript, JSON

# Initialize Modern MediaPipe Tasks Hand Landmarker
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

# Standard 21 Hand Skeleton Joint Connections
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (5, 9), (9, 10), (10, 11), (11, 12), # Middle
    (9, 13), (13, 14), (14, 15), (15, 16),# Ring
    (13, 17), (17, 18), (18, 19), (19, 20),# Pinky
    (0, 17)                              # Palm Base
]

def draw_skeleton(frame, landmarks, W, H):
    for p1, p2 in HAND_CONNECTIONS:
        x1, y1 = int(landmarks[p1].x * W), int(landmarks[p1].y * H)
        x2, y2 = int(landmarks[p2].x * W), int(landmarks[p2].y * H)
        cv2.line(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)
    for lm in landmarks:
        cx, cy = int(lm.x * W), int(lm.y * H)
        cv2.circle(frame, (cx, cy), 4, (0, 230, 115), -1)

# Motion & Smoothing state
prev_landmarks = None
smoothed_coords = None
active_pred = "..."
active_conf = 0.0
pending_pred = None
pending_count = 0
MOTION_THRESHOLD = 0.022

def process_frame(data_url):
    global prev_landmarks, smoothed_coords, active_pred, active_conf, pending_pred, pending_count
    
    try:
        if isinstance(data_url, str):
            if ',' in data_url:
                data_url = data_url.split(',', 1)[1]
            raw_bytes = base64.b64decode(data_url)
        else:
            raw_bytes = bytes(data_url)
        
        frame = cv2.imdecode(np.frombuffer(raw_bytes, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            return JSON({"error": "Could not decode frame"})
        
        # Flip horizontally for natural mirror selfie view
        frame = cv2.flip(frame, 1)
        H, W, _ = frame.shape
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        detection_result = detector.detect(mp_img)
        
        status_text = "None"
        
        if detection_result.hand_landmarks:
            landmarks = detection_result.hand_landmarks[0]
            
            # Draw skeleton connections & keypoints
            draw_skeleton(frame, landmarks, W, H)
            
            current_coords = np.array([[lm.x, lm.y] for lm in landmarks], dtype=np.float32)
            
            # 1. Motion detection
            if prev_landmarks is not None:
                disp = np.linalg.norm(current_coords - prev_landmarks, axis=1)
                motion_speed = float(np.mean(disp))
            else:
                motion_speed = 0.0
            prev_landmarks = current_coords.copy()
            is_moving = motion_speed > MOTION_THRESHOLD
            
            # 2. EMA smoothing
            if smoothed_coords is None:
                smoothed_coords = current_coords.copy()
            else:
                alpha = 0.70
                smoothed_coords = alpha * current_coords + (1.0 - alpha) * smoothed_coords
            
            # 3. Scale-invariant feature normalization
            xs = smoothed_coords[:, 0]
            ys = smoothed_coords[:, 1]
            min_x, min_y = float(np.min(xs)), float(np.min(ys))
            
            data_aux = []
            for i in range(21):
                data_aux.append(float(smoothed_coords[i, 0] - min_x))
                data_aux.append(float(smoothed_coords[i, 1] - min_y))
            data_aux = np.asarray(data_aux, dtype=np.float32)
            
            hand_scale = max(float(data_aux[0::2].max()), float(data_aux[1::2].max()))
            if hand_scale > 1e-4:
                data_aux /= hand_scale
            
            # 4. Model Prediction & Hysteresis
            if len(data_aux) == 42:
                # Ensure all tree estimators have monotonic_cst attribute in sklearn 1.4+
                if hasattr(model, 'estimators_'):
                    for est in model.estimators_:
                        if not hasattr(est, 'monotonic_cst'):
                            est.monotonic_cst = None
                
                proba = model.predict_proba([data_aux])[0]
                best_idx = int(np.argmax(proba))
                best_conf = float(proba[best_idx])
                candidate_label = class_names[best_idx]
                
                if candidate_label == active_pred:
                    active_conf = best_conf
                    pending_pred = None
                    pending_count = 0
                else:
                    if candidate_label == pending_pred:
                        pending_count += 1
                    else:
                        pending_pred = candidate_label
                        pending_count = 1
                    if pending_count >= 2 or best_conf > 0.75:
                        active_pred = candidate_label
                        active_conf = best_conf
                        pending_count = 0
                
                status_text = "Moving" if is_moving else "Steady"
                
                # Visual banner
                x1 = max(0, int(min_x * W) - 15)
                y1 = max(0, int(min_y * H) - 15)
                x2 = min(W, int(np.max(xs) * W) + 15)
                y2 = min(H, int(np.max(ys) * H) + 15)
                box_color = (0, 230, 115) if not is_moving else (255, 175, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                
                display_str = f"{active_pred} {active_conf*100:.0f}% [{status_text}]"
                cv2.putText(frame, display_str, (x1, max(25, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.75, box_color, 2, cv2.LINE_AA)
        else:
            prev_landmarks = None
            smoothed_coords = None
            active_pred = "..."
            active_conf = 0.0
            status_text = "No Hand"
        
        # Encode to JPEG base64 string
        _, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        b64_str = base64.b64encode(buffer).decode('utf-8')
        
        return JSON({
            "image": b64_str,
            "text": active_pred,
            "confidence": f"{active_conf*100:.1f}%",
            "status": status_text
        })
    except Exception as e:
        return JSON({"error": str(e)})

# Register callback for Google Colab
output.register_callback('notebook.process_frame', process_frame)
print("✅ Modern MediaPipe Tasks detector and callback successfully registered!")

In [ ]:
# Launch the interactive Colab webcam translator UI
display(HTML('''
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0f172a; color: white; padding: 20px; border-radius: 16px; max-width: 900px; margin: auto; box-shadow: 0 20px 25px -5px rgba(0, 0, 0, 0.5);">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 12px; margin-bottom: 16px;">
        <h2 style="margin: 0; font-size: 22px; font-weight: 800; color: #60a5fa;">🤟 Sign2Text <span style="font-size: 13px; color: #94a3b8; font-weight: 400;">Google Colab Live Feed</span></h2>
        <span id="statusBadge" style="background: #1e293b; color: #10b981; padding: 4px 12px; border-radius: 999px; font-size: 12px; font-weight: 600; border: 1px solid #059669;">Connecting Camera...</span>
    </div>

    <div style="display: flex; flex-wrap: wrap; gap: 20px;">
        <!-- Video Feed Container -->
        <div style="flex: 1 1 480px; min-width: 320px; position: relative;">
            <video id="video" width="480" height="360" autoplay playsinline style="display: none;"></video>
            <canvas id="inputCanvas" width="480" height="360" style="display: none;"></canvas>
            
            <div id="videoWrapper" style="position: relative; width: 480px; height: 360px; background: #1e293b; border-radius: 12px; overflow: hidden; border: 1px solid #334155; display: flex; align-items: center; justify-content: center;">
                <div id="loadingNotice" style="color: #94a3b8; text-align: center; font-size: 14px;">
                    <div style="font-size: 32px; margin-bottom: 8px;">📹</div>
                    Requesting camera permission...<br><span style="font-size: 12px; color: #64748b;">Click 'Allow' when your browser asks</span>
                </div>
                <img id="outputImg" width="480" height="360" style="display: none; border-radius: 12px; width: 100%; height: 100%; object-fit: cover;" />
            </div>
            <div style="margin-top: 8px; font-size: 12px; color: #94a3b8; text-align: center;">
                Live MediaPipe Tasks Hand Tracking (Flipped Mirror View)
            </div>
        </div>

        <!-- Prediction & Sentence Controls -->
        <div style="flex: 1 1 320px; min-width: 280px; display: flex; flex-direction: column; gap: 14px;">
            <!-- Prediction Card -->
            <div style="background: #1e293b; border: 1px solid #334155; border-radius: 12px; padding: 18px; text-align: center; position: relative;">
                <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #94a3b8; font-weight: 700; margin-bottom: 6px;">Live Prediction</div>
                <div id="predText" style="font-size: 52px; font-weight: 900; color: #38bdf8; min-height: 60px; line-height: 60px;">...</div>
                <div id="confText" style="font-size: 13px; font-family: monospace; color: #cbd5e1;">Confidence: --%</div>
            </div>

            <!-- Sentence Builder -->
            <div style="background: #1e293b; border: 1px solid #334155; border-radius: 12px; padding: 16px;">
                <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #94a3b8; font-weight: 700; margin-bottom: 8px;">Sentence Builder</div>
                <textarea id="sentenceBox" rows="3" style="width: 100%; box-sizing: border-box; background: #0f172a; border: 1px solid #475569; border-radius: 8px; color: white; padding: 10px; font-size: 14px; resize: none;" placeholder="Click 'Add Word' to build your sentence here..."></textarea>
                
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-top: 10px;">
                    <button id="addWordBtn" style="background: #334155; color: white; border: none; padding: 10px; border-radius: 8px; font-weight: 600; cursor: pointer;">+ Add Word</button>
                    <button id="speakBtn" style="background: #2563eb; color: white; border: none; padding: 10px; border-radius: 8px; font-weight: 600; cursor: pointer;">🔊 Speak</button>
                    <button id="clearBtn" style="grid-column: span 2; background: #1e293b; color: #ef4444; border: 1px solid #7f1d1d; padding: 8px; border-radius: 8px; font-weight: 500; cursor: pointer;">Clear Sentence</button>
                </div>
            </div>
        </div>
    </div>
</div>

<script>
(async function() {
    const video = document.getElementById('video');
    const canvas = document.getElementById('inputCanvas');
    const ctx = canvas.getContext('2d');
    const outputImg = document.getElementById('outputImg');
    const loadingNotice = document.getElementById('loadingNotice');
    const predText = document.getElementById('predText');
    const confText = document.getElementById('confText');
    const statusBadge = document.getElementById('statusBadge');
    const sentenceBox = document.getElementById('sentenceBox');

    // Access local webcam
    try {
        const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 480, height: 360 } });
        video.srcObject = stream;
        await video.play();
        statusBadge.textContent = "Stream Active";
        statusBadge.style.color = "#10b981";
        statusBadge.style.borderColor = "#059669";
    } catch (err) {
        statusBadge.textContent = "Camera Error";
        statusBadge.style.color = "#ef4444";
        statusBadge.style.borderColor = "#b91c1c";
        loadingNotice.innerHTML = '<span style="color:#ef4444">Camera access denied: ' + err.message + '</span>';
        return;
    }

    let isProcessing = false;

    // Sentence builder buttons
    document.getElementById('addWordBtn').onclick = () => {
        const current = predText.textContent.trim();
        if (current && current !== '...') {
            sentenceBox.value += (sentenceBox.value.length === 0 ? '' : ' ') + current;
        }
    };

    document.getElementById('clearBtn').onclick = () => {
        sentenceBox.value = '';
    };

    document.getElementById('speakBtn').onclick = () => {
        const text = sentenceBox.value.trim() || predText.textContent.trim();
        if (text && text !== '...') {
            const utterance = new SpeechSynthesisUtterance(text);
            utterance.rate = 0.95;
            window.speechSynthesis.speak(utterance);
        }
    };

    // Processing loop
    async function captureAndSend() {
        if (isProcessing) return;
        isProcessing = true;

        try {
            ctx.drawImage(video, 0, 0, 480, 360);
            const dataUrl = canvas.toDataURL('image/jpeg', 0.75);
            
            const result = await google.colab.kernel.invokeFunction('notebook.process_frame', [dataUrl], {});
            const resp = result.data['application/json'];
            
            if (resp && resp.image) {
                outputImg.src = 'data:image/jpeg;base64,' + resp.image;
                if (outputImg.style.display === 'none') {
                    loadingNotice.style.display = 'none';
                    outputImg.style.display = 'block';
                }
                predText.textContent = resp.text;
                confText.textContent = `Confidence: ${resp.confidence} (${resp.status})`;
            } else if (resp && resp.error) {
                console.error("Backend error:", resp.error);
                confText.textContent = `Error: ${resp.error}`;
            }
        } catch (e) {
            console.error("Processing error:", e);
        } finally {
            isProcessing = false;
            setTimeout(captureAndSend, 40);
        }
    }

    captureAndSend();
})();
</script>
'''))